In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,
    'axes.spines.top':False,'axes.spines.right':False,'font.size':11
})

# ── Dart board illustration ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
scenarios = [
    ('High Bias\nLow Variance', [(0.3,0.3),(0.35,0.28),(0.32,0.33),(0.28,0.31),(0.33,0.35),(0.29,0.29)], '#e85d20'),
    ('Low Bias\nHigh Variance', [(-0.1,0.4),(0.5,-0.3),(-0.4,-0.2),(0.3,0.5),(-0.2,-0.4),(0.4,0.1)], '#c0392b'),
    ('High Bias\nHigh Variance', [(0.4,0.3),(0.1,-0.4),(0.5,0.1),(-0.3,0.4),(0.3,-0.2),(0.4,0.5)], '#888'),
    ('Low Bias\nLow Variance ✓', [(0.05,0.03),(-0.04,0.06),(0.02,-0.05),(0.06,0.04),(-0.03,-0.02),(0.04,0.05)], '#1a7a45'),
]

for ax, (title, points, color) in zip(axes, scenarios):
    for r, alpha in [(1,'#f0f0f0'),(0.6,'#e0e8f0'),(0.3,'#c8d8f0'),(0.1,'#a0c0e8')]:
        circle = plt.Circle((0,0), r, color=alpha, zorder=1)
        ax.add_patch(circle)
    ax.plot(0, 0, 'k+', markersize=12, zorder=3, lw=2)
    xs, ys = zip(*points)
    ax.scatter(xs, ys, color=color, s=60, zorder=4, edgecolors='white', lw=0.5)
    ax.set_xlim(-1.2,1.2); ax.set_ylim(-1.2,1.2)
    ax.set_aspect('equal'); ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Bias-Variance as a Dart Board', fontsize=12, y=1.05)
plt.tight_layout()
plt.show()
print("📌 We want bottom-right: low bias (near bullseye) + low variance (tight cluster).")

# Load dataset
ads = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Advertising.csv')
print(f'✓ Advertising (ISLP): {ads.shape}')


In [ ]:
# ⚠️  Run this cell first before any others
# (Tip: Runtime → Run all  OR  Shift+Enter from the top)
import pandas as pd
ads = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Advertising.csv')
print(f"Advertising: {ads.shape}  |  Columns: {list(ads.columns)}")
ads.head()

In [ ]:
# ── Simulate the bias-variance tradeoff exactly ─────────────────────────────
np.random.seed(42)

def f_true(x): return np.sin(2*np.pi*x)

n_train = 30
n_reps  = 200
sigma   = 0.3
x_test  = np.linspace(0, 1, 100)

# Cap at degree 8 — degree 12 with n=30 is numerically unstable
# (more parameters than needed, leading to explosive extrapolation)
degrees = [1, 2, 3, 4, 5, 6, 7, 8]
bias2_list, var_list, total_list = [], [], []

for d in degrees:
    preds = []
    for _ in range(n_reps):
        x_tr = np.random.uniform(0, 1, n_train)
        y_tr = f_true(x_tr) + np.random.normal(0, sigma, n_train)
        pipe = Pipeline([('poly', PolynomialFeatures(d)), ('lr', LinearRegression())])
        pipe.fit(x_tr.reshape(-1,1), y_tr)
        preds.append(pipe.predict(x_test.reshape(-1,1)))

    preds    = np.array(preds)
    mean_pred = preds.mean(axis=0)

    bias2    = np.mean((mean_pred - f_true(x_test))**2)
    variance = np.mean(preds.var(axis=0))
    irred    = sigma**2

    bias2_list.append(bias2)
    var_list.append(variance)
    total_list.append(bias2 + variance + irred)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: linear scale — sufficient since values stay reasonable up to degree 8
axes[0].plot(degrees, bias2_list, 'o-', color='#e85d20', lw=2, label='Bias²', markersize=6)
axes[0].plot(degrees, var_list,   's-', color='#1e5fa8', lw=2, label='Variance', markersize=6)
axes[0].axhline(sigma**2, color='#888', lw=1.5, ls=':', label=f'Irreducible ({sigma**2:.2f})')
axes[0].set_xlabel('Model Complexity (degree)')
axes[0].set_ylabel('Error Component')
axes[0].set_title('Bias² falls, Variance rises with complexity')
axes[0].legend()

# Right: U-shaped total test error
best_d = degrees[np.argmin(total_list)]
axes[1].plot(degrees, total_list, 'o-', color='#1a7a45', lw=2.5, label='Total Test Error', markersize=6)
axes[1].axvline(best_d, color='#e85d20', lw=1.5, ls='--', label=f'Sweet spot (degree {best_d})')
axes[1].set_xlabel('Model Complexity (degree)')
axes[1].set_ylabel('Expected Test MSE')
axes[1].set_title('U-shaped Test Error Curve')
axes[1].legend()

plt.suptitle('The Bias-Variance Tradeoff', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n📌 Sweet spot at degree {best_d}.")
print(f"   Degree 1 — high bias  : Bias²={bias2_list[0]:.3f}  Var={var_list[0]:.4f}")
print(f"   Degree {best_d} — balanced : Bias²={bias2_list[best_d-1]:.3f}  Var={var_list[best_d-1]:.4f}")
print(f"   Degree 8 — high var   : Bias²={bias2_list[-1]:.3f}  Var={var_list[-1]:.4f}")
print(f"\n   Why cap at degree 8? With only {n_train} training points, degree 12")
print(f"   fits 13 coefficients to 30 points — numerical instability dominates.")
print(f"   The tradeoff pattern is identical and far clearer at lower degrees.")


In [ ]:
# ── Learning curves for underfit, good fit, overfit ─────────────────────────
np.random.seed(1)
n_total = 200
X_full = np.random.uniform(0, 1, n_total)
Y_full = f_true(X_full) + np.random.normal(0, sigma, n_total)

train_sizes = range(10, 160, 10)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
model_specs = [(1, 'Degree 1 — Underfit (High Bias)'),
               (3, 'Degree 3 — Good Fit'),
               (12, 'Degree 12 — Overfit (High Variance)')]
model_colors = ['#e85d20', '#1a7a45', '#c0392b']

for ax, (deg, title), color in zip(axes, model_specs, model_colors):
    tr_errs, val_errs = [], []
    for n in train_sizes:
        X_tr, Y_tr = X_full[:n], Y_full[:n]
        X_val, Y_val = X_full[160:], Y_full[160:]
        pipe = Pipeline([('poly',PolynomialFeatures(deg)),('lr',LinearRegression())])
        pipe.fit(X_tr.reshape(-1,1), Y_tr)
        tr_errs.append(mean_squared_error(Y_tr, pipe.predict(X_tr.reshape(-1,1))))
        val_errs.append(mean_squared_error(Y_val, pipe.predict(X_val.reshape(-1,1))))
    
    ax.plot(list(train_sizes), tr_errs,  '-', color=color,    lw=2, label='Training error')
    ax.plot(list(train_sizes), val_errs, '--', color='#1e5fa8', lw=2, label='Validation error')
    ax.axhline(sigma**2, color='#aaa', lw=1, ls=':', label='Irreducible floor')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Training set size')
    if ax == axes[0]: ax.set_ylabel('MSE')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 0.35)

plt.suptitle('Learning Curves — Diagnosing Bias vs Variance', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("\n📌 High bias (left): both curves meet at high error — model can't learn the pattern.")
print("   Good fit (middle): curves converge near irreducible floor.")
print("   High variance (right): big gap between training and validation — needs more data or regularization.")

In [ ]:
# @title 📝 Quiz — Bias-Variance Tradeoff
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** As model complexity increases, what happens to bias?
# @markdown **Q2:** As model complexity increases, what happens to variance?
# @markdown **Q3:** What does the irreducible error floor represent?
# @markdown **Q4:** A model with high training accuracy but poor test accuracy suffers from what?
# @markdown **Q5:** Name one practical way to reduce variance without changing the model family

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

In [ ]:
_NB_NAME_="Bias-Variance Tradeoff"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*